# LeetCode 323 — Number of Connected Components in an Undirected Graph bằng Python

Link đề: https://leetcode.com/problems/number-of-connected-components-in-an-undirected-graph/ · Độ khó: **Medium** (Premium) · Chủ đề: Graph, Union Find, DFS, BFS

Skeleton LeetCode:

```python
class Solution:
    def countComponents(self, n: int, edges: List[List[int]]) -> int:
        ...
```

Ghi chú: đề bài viết theo hiểu biết về bài, nên đối chiếu lại nếu có tài khoản Premium; ví dụ và constraints lấy từ file đề có sẵn. Mọi đáp án trong notebook đều được kiểm chứng bằng code và đối chiếu với brute force độc lập (bao đóng bắc cầu Floyd–Warshall rồi đếm các lớp tương đương).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: đồ thị vô hướng `n` nút (đánh số `0..n-1`) và danh sách cạnh; đếm **số thành phần liên thông** (các "đảo" nút nối được với nhau qua cạnh; **nút cô lập** cũng là một thành phần).
2. Nắm ba công cụ nền tảng của đồ thị: **danh sách kề**, **DFS/BFS** (duyệt) và **Union-Find (DSU)** (gộp tập), cùng ý nghĩa "**mỗi lần gộp hai thành phần khác nhau, số thành phần giảm đi 1**".
3. Đi từ **bao đóng bắc cầu** `O(n³)` → **DSU có nén đường đi** → **DSU gộp theo kích thước** → **DFS (lặp)** → **BFS**, mỗi cách đếm thành phần theo cách riêng.
4. Hiểu công thức `số thành phần = n − số cạnh gộp thành công`, và vì sao `n − len(edges)` **chỉ đúng khi không có chu trình**.
5. Tránh các lỗi kinh điển: **quên nút cô lập** (chỉ duyệt các nút xuất hiện trong cạnh), thêm cạnh **một chiều**, **không đánh dấu đã thăm**, cho `count` giảm cả khi `find(a) == find(b)`, và `find` đệ quy bị **tràn**.

## 2. Đề bài & ví dụ

Cho một đồ thị gồm `n` nút được đánh nhãn `0` đến `n - 1`, và mảng `edges` với `edges[i] = [a_i, b_i]` nghĩa là có **cạnh vô hướng** giữa `a_i` và `b_i`. Trả về **số thành phần liên thông** của đồ thị.

**Ví dụ 1**

- Input: `n = 5, edges = [[0,1],[1,2],[3,4]]`
- Output: `2`
- Giải thích: thành phần `{0,1,2}` và thành phần `{3,4}`.

**Ví dụ 2**

- Input: `n = 5, edges = [[0,1],[1,2],[2,3],[3,4]]`
- Output: `1`

**Ràng buộc:** `1 <= n <= 2000`; `1 <= edges.length <= 5000`; `edges[i] = [a_i, b_i]`; `a_i != b_i`; không có cạnh lặp.

In [1]:
from typing import List
from collections import deque, defaultdict
import random
import time

examples = [
    (5, [[0, 1], [1, 2], [3, 4]], 2),
    (5, [[0, 1], [1, 2], [2, 3], [3, 4]], 1),
]
for n, edges, expected in examples:
    print(f"n={n}, edges={edges} -> kỳ vọng {expected}")

n=5, edges=[[0, 1], [1, 2], [3, 4]] -> kỳ vọng 2
n=5, edges=[[0, 1], [1, 2], [2, 3], [3, 4]] -> kỳ vọng 1


## 3. Kiến thức nền

### 3.1. Thành phần liên thông

Hai nút **liên thông** nếu có đường đi (qua các cạnh) giữa chúng. "Liên thông" là quan hệ **tương đương** (phản xạ, đối xứng, bắc cầu) nên chia các nút thành các **lớp** rời nhau — đó là các thành phần liên thông. Nút không có cạnh nào là một thành phần có đúng một nút.
#### Code demo

In [2]:
n, edges = 6, [[0, 1], [1, 2], [3, 4]]
print("các thành phần mong đợi: {0,1,2}, {3,4}, {5} -> tổng", 3, "(nút 5 cô lập vẫn tính)")

các thành phần mong đợi: {0,1,2}, {3,4}, {5} -> tổng 3 (nút 5 cô lập vẫn tính)


### 3.2. Danh sách kề (vô hướng ⇒ thêm hai chiều)

`adj[v]` chứa các hàng xóm của `v`. Với cạnh `[a, b]` phải thêm **cả** `adj[a].append(b)` **và** `adj[b].append(a)`.
#### Code demo

In [3]:
adj = [[] for _ in range(5)]
for a, b in [[0, 1], [1, 2], [3, 4]]:
    adj[a].append(b)
    adj[b].append(a)
print(adj)          # [[1], [0, 2], [1], [4], [3]]

[[1], [0, 2], [1], [4], [3]]


### 3.3. Duyệt DFS/BFS và đếm thành phần

Duyệt qua **mọi nút** `v = 0..n-1`: nếu `v` **chưa thăm** thì đó là một **thành phần mới** (`count += 1`) và ta duyệt (DFS hoặc BFS) để đánh dấu **toàn bộ** thành phần của `v` là đã thăm. Vòng lặp ngoài **phải đi qua mọi nút** (không chỉ nút có cạnh) để không bỏ sót nút cô lập.
#### Code demo

In [4]:
def dfs_mark(start, adj, visited):
    stack = [start]
    visited[start] = True
    while stack:
        v = stack.pop()
        for u in adj[v]:
            if not visited[u]:
                visited[u] = True
                stack.append(u)


visited = [False] * 5
components = 0
for v in range(5):
    if not visited[v]:
        components += 1
        dfs_mark(v, adj, visited)
        print(f"bắt đầu thành phần mới từ nút {v}; visited = {visited}")
print("số thành phần:", components)

bắt đầu thành phần mới từ nút 0; visited = [True, True, True, False, False]
bắt đầu thành phần mới từ nút 3; visited = [True, True, True, True, True]
số thành phần: 2


### 3.4. Union-Find (DSU)

`parent[x]` trỏ lên đại diện; `find(x)` tìm gốc (có **nén đường đi**); `union(a, b)` nối gốc của `a` vào gốc của `b` nếu khác nhau. Khởi đầu có `n` tập đơn (`n` thành phần). Mỗi lần **gộp thành công** hai tập khác nhau, số thành phần **giảm đúng `1`**.
#### Code demo

In [5]:
parent = list(range(5))


def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]               # nén đường đi (path halving)
        x = parent[x]
    return x


count = 5
for a, b in [[0, 1], [1, 2], [3, 4], [0, 2]]:
    ra, rb = find(a), find(b)
    if ra != rb:
        parent[ra] = rb
        count -= 1
    print(f"cạnh ({a},{b}): gốc {ra} và {rb} -> {'gộp' if ra != rb else 'cùng thành phần (bỏ qua)'}; số thành phần = {count}")

cạnh (0,1): gốc 0 và 1 -> gộp; số thành phần = 4
cạnh (1,2): gốc 1 và 2 -> gộp; số thành phần = 3
cạnh (3,4): gốc 3 và 4 -> gộp; số thành phần = 2
cạnh (0,2): gốc 2 và 2 -> cùng thành phần (bỏ qua); số thành phần = 2


### 3.5. Cạnh thừa không giảm số thành phần

Một cạnh nối hai nút **đã** liên thông (tạo chu trình) **không** làm giảm số thành phần. Vì vậy công thức `n − len(edges)` **sai** khi có chu trình; đúng là `n − (số lần gộp thành công)`.
#### Code demo

In [6]:
n, tri = 3, [[0, 1], [1, 2], [0, 2]]
print("n - len(edges) =", n - len(tri), "| thực tế số thành phần = 1 (tam giác liên thông)")

n - len(edges) = 0 | thực tế số thành phần = 1 (tam giác liên thông)


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Ví dụ 1: `n = 5`, cạnh `0-1`, `1-2`, `3-4`.

- Ban đầu `5` thành phần đơn: `{0}, {1}, {2}, {3}, {4}`.
- Thêm `0-1` ⇒ gộp `{0},{1}`: còn `4`.
- Thêm `1-2` ⇒ gộp `{0,1},{2}`: còn `3`.
- Thêm `3-4` ⇒ gộp `{3},{4}`: còn `2`.

Đáp án `2`. **Cách nghĩ:** bắt đầu từ "mỗi nút một thành phần", mỗi cạnh hoặc gộp hai thành phần khác nhau (số thành phần giảm `1`) hoặc không làm gì (cạnh thừa).

## 5. Cách 1 — Brute Force / baseline: bao đóng bắc cầu (Floyd–Warshall) rồi đếm lớp

**Ý tưởng.** Dựng ma trận `reach[i][j]` (có đường đi từ `i` tới `j`) bằng Floyd–Warshall boolean; hai nút cùng thành phần ⟺ `reach[i][j]`. Đếm số nút "đại diện" (nút nhỏ nhất của lớp): `i` là đại diện nếu không có `j < i` với `reach[j][i]`.
#### Code demo

In [7]:
def count_closure(n, edges):
    reach = [[i == j for j in range(n)] for i in range(n)]
    for a, b in edges:
        reach[a][b] = reach[b][a] = True
    for k in range(n):
        for i in range(n):
            if reach[i][k]:
                for j in range(n):
                    if reach[k][j]:
                        reach[i][j] = True
    return sum(1 for i in range(n) if not any(reach[j][i] for j in range(i)))


for n, edges, expected in examples:
    print(n, edges, "->", count_closure(n, edges), "(kỳ vọng", expected, ")")

5 [[0, 1], [1, 2], [3, 4]] -> 2 (kỳ vọng 2 )
5 [[0, 1], [1, 2], [2, 3], [3, 4]] -> 1 (kỳ vọng 1 )


**Độ phức tạp (Cách 1).** **Time `O(n³)`** (với `n = 2000` là `8·10^9` — không chạy được), **Space `O(n²)`**. Chỉ làm oracle cho `n` nhỏ.

**Nút thắt.** Ta tính liên thông của **mọi cặp** trong khi chỉ cần đếm lớp. DSU/DFS làm trong thời gian gần tuyến tính theo `n + số cạnh`.

## 6. Ý tưởng thuật toán / bước đột phá

**Hai hướng gần tuyến tính:**

- **DSU:** `count = n`; với mỗi cạnh, nếu hai gốc khác nhau thì `union` và `count -= 1`. Đáp án `count`.
- **Duyệt đồ thị:** với mỗi nút chưa thăm, tăng bộ đếm và đánh dấu cả thành phần bằng DFS/BFS.

Cả hai `O(n + E)` (DSU: `O(E·α(n))`). DSU hợp khi cạnh đến **dần dần** (online); DFS/BFS hợp khi cần biết **các nút của mỗi thành phần**.

## 7. Các cách tiếp theo

### Cách 2 — Union-Find có nén đường đi (cách nộp chuẩn)

**Giải thích.** Như mục 3.4; trả về biến đếm.
#### Code demo

In [8]:
def count_dsu(n, edges):
    parent = list(range(n))

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    count = n
    for a, b in edges:
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[ra] = rb
            count -= 1                                     # chỉ giảm khi gộp thành công
    return count


for n, edges, expected in examples:
    print(n, edges, "->", count_dsu(n, edges), "(kỳ vọng", expected, ")")

5 [[0, 1], [1, 2], [3, 4]] -> 2 (kỳ vọng 2 )
5 [[0, 1], [1, 2], [2, 3], [3, 4]] -> 1 (kỳ vọng 1 )


In [9]:
# Code nộp LeetCode cho Cách 2
class Solution:
    def countComponents(self, n: int, edges: List[List[int]]) -> int:
        def find(x: int) -> int:
            if p[x] != x:
                p[x] = find(p[x])
            return p[x]

        p = list(range(n))
        for a, b in edges:
            pa, pb = find(a), find(b)
            if pa != pb:
                p[pa] = pb
                n -= 1
        return n


print(Solution().countComponents(5, [[0, 1], [1, 2], [3, 4]]))     # 2

2


**Độ phức tạp Cách 2:** **Time `O(E · α(n))`** ≈ `O(E)`, **Space `O(n)`**. (Bản nộp dùng `find` đệ quy như LeetCode; ở phần lỗi ta sẽ thấy vì sao `find` lặp an toàn hơn khi chuỗi dài.)

### Cách 3 — DSU gộp theo kích thước (union by size)

**Giải thích.** Luôn nối cây **nhỏ** vào cây **lớn** để chiều cao `O(log n)`; kết hợp nén đường đi cho gần `O(1)`. Cũng cho biết **kích thước** mỗi thành phần.
#### Code demo

In [10]:
def count_dsu_size(n, edges):
    parent = list(range(n))
    size = [1] * n

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    count = n
    for a, b in edges:
        ra, rb = find(a), find(b)
        if ra == rb:
            continue
        if size[ra] < size[rb]:
            ra, rb = rb, ra                                # ra luôn là cây lớn hơn
        parent[rb] = ra
        size[ra] += size[rb]
        count -= 1
    return count


for n, edges, expected in examples:
    print(n, edges, "->", count_dsu_size(n, edges), "(kỳ vọng", expected, ")")

5 [[0, 1], [1, 2], [3, 4]] -> 2 (kỳ vọng 2 )
5 [[0, 1], [1, 2], [2, 3], [3, 4]] -> 1 (kỳ vọng 1 )


**Độ phức tạp Cách 3:** **Time `O(E · α(n))`**, **Space `O(n)`**.

### Cách 4 — DFS (lặp) trên danh sách kề

**Giải thích.** Như mục 3.3.
#### Code demo

In [11]:
def count_dfs(n, edges):
    adj = [[] for _ in range(n)]
    for a, b in edges:
        adj[a].append(b)
        adj[b].append(a)
    visited = [False] * n
    components = 0
    for start in range(n):                                  # đi qua MỌI nút, kể cả nút cô lập
        if visited[start]:
            continue
        components += 1
        visited[start] = True
        stack = [start]
        while stack:
            v = stack.pop()
            for u in adj[v]:
                if not visited[u]:
                    visited[u] = True
                    stack.append(u)
    return components


for n, edges, expected in examples:
    print(n, edges, "->", count_dfs(n, edges), "(kỳ vọng", expected, ")")

5 [[0, 1], [1, 2], [3, 4]] -> 2 (kỳ vọng 2 )
5 [[0, 1], [1, 2], [2, 3], [3, 4]] -> 1 (kỳ vọng 1 )


**Độ phức tạp Cách 4:** **Time `O(n + E)`**, **Space `O(n + E)`** (danh sách kề, stack, `visited`).

### Cách 5 — BFS với hàng đợi

**Giải thích.** Giống Cách 4 nhưng dùng `deque` và `popleft`; mỗi lần gặp nút chưa thăm ở vòng ngoài là một thành phần mới.
#### Code demo

In [12]:
def count_bfs(n, edges):
    adj = defaultdict(list)
    for a, b in edges:
        adj[a].append(b)
        adj[b].append(a)
    seen = set()
    components = 0
    for start in range(n):
        if start in seen:
            continue
        components += 1
        seen.add(start)
        queue = deque([start])
        while queue:
            v = queue.popleft()
            for u in adj[v]:
                if u not in seen:
                    seen.add(u)
                    queue.append(u)
    return components


for n, edges, expected in examples:
    print(n, edges, "->", count_bfs(n, edges), "(kỳ vọng", expected, ")")

5 [[0, 1], [1, 2], [3, 4]] -> 2 (kỳ vọng 2 )
5 [[0, 1], [1, 2], [2, 3], [3, 4]] -> 1 (kỳ vọng 1 )


**Độ phức tạp Cách 5:** **Time `O(n + E)`**, **Space `O(n + E)`**.

> Cách 2 (DSU) là bản nộp tiêu chuẩn; Cách 4/5 cho thấy DFS/BFS; Cách 3 là DSU tối ưu hơn.

## 8. TRACE từng bước

Ta trace DSU trên `n = 6`, `edges = [[0,1],[2,3],[1,3],[3,0],[4,5]]` (cạnh `3-0` là cạnh thừa vì `0` và `3` đã liên thông qua `0-1-3`) và DFS đếm thành phần.
#### Code demo

In [13]:
def trace_dsu(n, edges):
    parent = list(range(n))

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    count = n
    print(f"n={n}, ban đầu {count} thành phần")
    for a, b in edges:
        ra, rb = find(a), find(b)
        if ra == rb:
            print(f"  cạnh ({a},{b}): cùng gốc {ra} -> cạnh thừa, số thành phần vẫn {count}")
        else:
            parent[ra] = rb
            count -= 1
            print(f"  cạnh ({a},{b}): gốc {ra} != {rb} -> gộp, còn {count}; parent={parent}")
    print("kết quả:", count)
    return count


trace_dsu(6, [[0, 1], [2, 3], [1, 3], [3, 0], [4, 5]])

n=6, ban đầu 6 thành phần
  cạnh (0,1): gốc 0 != 1 -> gộp, còn 5; parent=[1, 1, 2, 3, 4, 5]
  cạnh (2,3): gốc 2 != 3 -> gộp, còn 4; parent=[1, 1, 3, 3, 4, 5]
  cạnh (1,3): gốc 1 != 3 -> gộp, còn 3; parent=[1, 3, 3, 3, 4, 5]
  cạnh (3,0): cùng gốc 3 -> cạnh thừa, số thành phần vẫn 3
  cạnh (4,5): gốc 4 != 5 -> gộp, còn 2; parent=[3, 3, 3, 3, 5, 5]
kết quả: 2


2

In [14]:
def trace_dfs(n, edges):
    adj = [[] for _ in range(n)]
    for a, b in edges:
        adj[a].append(b)
        adj[b].append(a)
    visited = [False] * n
    count = 0
    for s in range(n):
        if visited[s]:
            print(f"nút {s}: đã thăm, bỏ qua")
            continue
        count += 1
        comp = []
        visited[s] = True
        stack = [s]
        while stack:
            v = stack.pop()
            comp.append(v)
            for u in adj[v]:
                if not visited[u]:
                    visited[u] = True
                    stack.append(u)
        print(f"nút {s}: chưa thăm -> thành phần #{count} = {sorted(comp)}")
    return count


trace_dfs(6, [[0, 1], [2, 3], [1, 3], [3, 0], [4, 5]])
print()
trace_dfs(4, [[0, 1]])

nút 0: chưa thăm -> thành phần #1 = [0, 1, 2, 3]
nút 1: đã thăm, bỏ qua
nút 2: đã thăm, bỏ qua
nút 3: đã thăm, bỏ qua
nút 4: chưa thăm -> thành phần #2 = [4, 5]
nút 5: đã thăm, bỏ qua

nút 0: chưa thăm -> thành phần #1 = [0, 1]
nút 1: đã thăm, bỏ qua
nút 2: chưa thăm -> thành phần #2 = [2]
nút 3: chưa thăm -> thành phần #3 = [3]


3

Bảng tóm tắt DSU (`n = 6`): `6 → 5 (0-1) → 4 (2-3) → 3 (1-3 gộp {0,1} với {2,3}) → 3 (3-0: thừa) → 2 (4-5)`. Đáp án `2`.

## 9. Code nộp LeetCode

Bản nộp dùng DSU. Đúng `class Solution`, đúng `countComponents(self, n: int, edges: List[List[int]]) -> int`. Ở đây ta dùng `find` lặp (an toàn hơn khi đồ thị là một chuỗi dài).
#### Code demo

In [15]:
class Solution:
    def countComponents(self, n: int, edges: List[List[int]]) -> int:
        parent = list(range(n))

        def find(x: int) -> int:
            while parent[x] != x:                 # tìm gốc, nén đường đi (path halving)
                parent[x] = parent[parent[x]]
                x = parent[x]
            return x

        components = n                            # mỗi nút ban đầu là một thành phần
        for a, b in edges:
            ra, rb = find(a), find(b)
            if ra != rb:                          # chỉ khi gộp hai thành phần khác nhau
                parent[ra] = rb
                components -= 1
        return components


sol = Solution()
for n, edges, expected in examples:
    assert sol.countComponents(n, edges) == expected
print("OK: 2 ví dụ của đề")

OK: 2 ví dụ của đề


## 10. Vì sao thuật toán đúng

- **DSU:** bất biến — tại mọi thời điểm, hai nút có cùng gốc ⟺ chúng liên thông qua các cạnh đã xử lý, và `components` bằng số gốc phân biệt. Ban đầu `n` gốc. Xử lý cạnh `(a, b)`: nếu `find(a) != find(b)` thì gộp hai tập ⇒ số gốc giảm `1`, đúng; nếu bằng nhau thì không đổi (cạnh thừa). Sau mọi cạnh, số gốc = số thành phần liên thông.
- **DFS/BFS:** từ nút chưa thăm `s`, duyệt đánh dấu đúng tập nút liên thông với `s` (mọi nút tới được qua cạnh); mỗi thành phần được bắt đầu đúng một lần (lần đầu gặp nút nào đó của nó ở vòng ngoài). Vòng ngoài đi qua mọi `0..n-1` nên nút cô lập được đếm.
- **Bao đóng bắc cầu:** `reach[i][j]` đúng ⟺ có đường đi; đại diện của lớp là nút nhỏ nhất ⇒ đếm lớp đúng.
- **Công thức:** `components = n − (số cạnh nối hai thành phần khác nhau)`.

## 11. Phân tích Time / Space Complexity

Gọi `V = n` (≤ 2000), `E = len(edges)` (≤ 5000).

| Cách | Time | Space |
|---|---|---|
| 1. Bao đóng bắc cầu | `O(V³)` | `O(V²)` |
| 2. DSU nén đường đi | `O(E·α(V))` | `O(V)` |
| 3. DSU theo kích thước | `O(E·α(V))` | `O(V)` |
| 4. DFS lặp | `O(V + E)` | `O(V + E)` |
| 5. BFS | `O(V + E)` | `O(V + E)` |

Với `V = 2000`, `E = 5000`: cỡ `10^4` phép — tức thời. Cách 1 là `8·10^9`. Không có khác biệt average/worst về bậc.

## 12. Bộ test

`oracle` là brute force độc lập: **bao đóng bắc cầu** (như Cách 1) nhưng đếm lớp theo cách khác: gom các hàng `reach[i]` thành **tập các tuple duy nhất** (mỗi thành phần cho cùng một hàng). Dùng cho `n <= 9`.
#### Code demo

In [16]:
def oracle(n, edges):
    reach = [[i == j for j in range(n)] for i in range(n)]
    for a, b in edges:
        reach[a][b] = reach[b][a] = True
    changed = True
    while changed:                                               # lặp tới điểm cố định (khác Floyd–Warshall)
        changed = False
        for i in range(n):
            for j in range(n):
                if not reach[i][j] and any(reach[i][k] and reach[k][j] for k in range(n)):
                    reach[i][j] = True
                    changed = True
    return len({tuple(row) for row in reach})                   # mỗi thành phần ứng với một hàng duy nhất


impls = {
    "closure": count_closure,
    "dsu": count_dsu,
    "dsu_size": count_dsu_size,
    "dfs": count_dfs,
    "bfs": count_bfs,
    "submit": sol.countComponents,
}

for n, edges, expected in examples:
    assert oracle(n, edges) == expected
    for key, fn in impls.items():
        assert fn(n, [list(e) for e in edges]) == expected, (key, n, edges, fn(n, edges), expected)
print("PASS ví dụ của đề (mọi cách và oracle)")

PASS ví dụ của đề (mọi cách và oracle)


### 12.1. Các trường hợp biên
#### Code demo

In [17]:
edge_cases = [
    (1, [], 1, "một nút, không cạnh"),
    (2, [], 2, "hai nút không cạnh"),
    (2, [[0, 1]], 1, "hai nút một cạnh"),
    (5, [], 5, "toàn nút cô lập"),
    (4, [[0, 1], [2, 3]], 2, "hai cặp"),
    (3, [[0, 1], [1, 2], [0, 2]], 1, "tam giác (chu trình): n - len(edges) = 0 là sai"),
    (6, [[0, 1], [1, 2], [3, 4]], 3, "có nút cô lập 5"),
    (5, [[0, 1], [0, 2], [0, 3], [0, 4]], 1, "sao"),
    (7, [[0, 1], [2, 3], [4, 5]], 4, "ba cặp và một nút cô lập"),
    (6, [[5, 4], [4, 3], [3, 2], [2, 1]], 2, "đường đi bị đảo hướng, nút 0 cô lập"),
]
for n, edges, expected, label in edge_cases:
    assert oracle(n, edges) == expected, (label, oracle(n, edges), expected)
    for key, fn in impls.items():
        assert fn(n, [list(e) for e in edges]) == expected, (key, label, fn(n, edges), expected)
    print("PASS", label)

PASS một nút, không cạnh
PASS hai nút không cạnh
PASS hai nút một cạnh
PASS toàn nút cô lập
PASS hai cặp
PASS tam giác (chu trình): n - len(edges) = 0 là sai
PASS có nút cô lập 5
PASS sao
PASS ba cặp và một nút cô lập
PASS đường đi bị đảo hướng, nút 0 cô lập


### 12.2. Tính chất và phản ví dụ

- `1 <= đáp án <= n`; không có cạnh thì `= n`; đồ thị liên thông thì `= 1`.
- **Bất biến** khi xáo trộn thứ tự cạnh, đảo chiều từng cạnh, đổi nhãn nút.
- Thêm một cạnh làm số thành phần **giảm 1 hoặc giữ nguyên** (không bao giờ tăng).
- **Phản ví dụ** cho công thức `n − len(edges)`: tam giác cho `0` (sai).
#### Code demo

In [18]:
rng = random.Random(323)


def random_graph(n, m, rng):
    pairs = [(a, b) for a in range(n) for b in range(a + 1, n)]
    chosen = rng.sample(pairs, min(m, len(pairs)))
    return [list(e) if rng.random() < 0.5 else [e[1], e[0]] for e in chosen]


for _ in range(300):
    n = rng.randint(1, 15)
    edges = random_graph(n, rng.randint(0, 20), rng)
    ans = sol.countComponents(n, [e[:] for e in edges])
    assert 1 <= ans <= n
    shuffled = [e[::-1] for e in edges]
    rng.shuffle(shuffled)
    assert sol.countComponents(n, shuffled) == ans
    relabel = list(range(n))
    rng.shuffle(relabel)
    assert sol.countComponents(n, [[relabel[a], relabel[b]] for a, b in edges]) == ans
    all_pairs = {tuple(sorted(e)) for e in edges}
    cand = [(a, b) for a in range(n) for b in range(a + 1, n) if (a, b) not in all_pairs]
    if cand:
        extra = list(rng.choice(cand))
        more = sol.countComponents(n, [e[:] for e in edges] + [extra])
        assert more in (ans, ans - 1)
print("PASS tính chất (cận, bất biến thứ tự/hướng/nhãn, thêm cạnh giảm ≤ 1) trên 300 đồ thị ngẫu nhiên")

tri = [[0, 1], [1, 2], [0, 2]]
print("tam giác: đúng", sol.countComponents(3, tri), "| n - len(edges):", 3 - len(tri))
assert 3 - len(tri) != sol.countComponents(3, tri)

PASS tính chất (cận, bất biến thứ tự/hướng/nhãn, thêm cạnh giảm ≤ 1) trên 300 đồ thị ngẫu nhiên
tam giác: đúng 1 | n - len(edges): 0


### 12.3. Đối chiếu ngẫu nhiên (có seed) và đồ thị lớn
#### Code demo

In [19]:
rng = random.Random(4323)
for trial in range(2000):
    n = rng.randint(1, 9)
    edges = random_graph(n, rng.randint(0, 12), rng)
    truth = oracle(n, edges)
    for key, fn in impls.items():
        assert fn(n, [e[:] for e in edges]) == truth, (key, n, edges, fn(n, edges), truth)
print("PASS 2000 đồ thị ngẫu nhiên nhỏ: mọi cách khớp oracle")

# Đồ thị cỡ đề: n = 2000, 5000 cạnh; và các dạng đặc biệt
cases = []
cases.append(("ngẫu nhiên thưa", 2000, random_graph(2000, 800, rng)))
cases.append(("ngẫu nhiên 5000 cạnh", 2000, random_graph(2000, 5000, rng)))
cases.append(("đường đi dài", 2000, [[i, i + 1] for i in range(1999)]))
cases.append(("đường đi đảo thứ tự", 2000, [[i, i + 1] for i in range(1998, -1, -1)]))
cases.append(("sao", 2000, [[0, i] for i in range(1, 2000)]))
cases.append(("1000 cặp", 2000, [[2 * i, 2 * i + 1] for i in range(1000)]))
for label, n, edges in cases:
    row, ref = [], None
    for key in ("dsu", "dsu_size", "dfs", "bfs", "submit"):
        t0 = time.perf_counter()
        value = impls[key](n, [e[:] for e in edges])
        dt = (time.perf_counter() - t0) * 1000
        ref = value if ref is None else ref
        assert value == ref, (label, key)
        row.append(f"{key}={dt:5.1f}ms")
    print(f"{label:<22} -> {ref:>5} thành phần | " + " ".join(row))
print("PASS đồ thị lớn")

PASS 2000 đồ thị ngẫu nhiên nhỏ: mọi cách khớp oracle


ngẫu nhiên thưa        ->  1200 thành phần | dsu=  0.4ms dsu_size=  0.3ms dfs=  0.5ms bfs=  1.8ms submit=  0.3ms
ngẫu nhiên 5000 cạnh   ->    15 thành phần | dsu= 18.4ms dsu_size=  1.9ms dfs=  1.6ms bfs=  2.1ms submit=  2.1ms
đường đi dài           ->     1 thành phần | dsu=  0.4ms dsu_size=  0.5ms dfs=  0.5ms bfs=  0.6ms submit=  0.3ms
đường đi đảo thứ tự    ->     1 thành phần | dsu=  0.4ms dsu_size=  0.7ms dfs=  0.5ms bfs=  0.6ms submit=  0.4ms
sao                    ->     1 thành phần | dsu=  0.5ms dsu_size=  0.4ms dfs=  0.5ms bfs=  0.8ms submit=  0.5ms
1000 cặp               ->  1000 thành phần | dsu=  0.2ms dsu_size=  0.2ms dfs=  0.4ms bfs=  0.6ms submit=  0.2ms
PASS đồ thị lớn


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Quên nút cô lập (chỉ duyệt các nút xuất hiện trong cạnh)
#### Code sai

In [20]:
def sai_quen_nut_co_lap(n, edges):
    adj = defaultdict(list)
    for a, b in edges:
        adj[a].append(b)
        adj[b].append(a)
    seen, count = set(), 0
    for start in list(adj.keys()):                         # SAI: chỉ các nút có cạnh
        if start in seen:
            continue
        count += 1
        stack = [start]
        seen.add(start)
        while stack:
            v = stack.pop()
            for u in adj[v]:
                if u not in seen:
                    seen.add(u)
                    stack.append(u)
    return count


for n, edges in [(6, [[0, 1], [1, 2], [3, 4]]), (4, [[0, 1]])]:
    print(n, edges, "đúng:", sol.countComponents(n, edges), "| sai:", sai_quen_nut_co_lap(n, edges))

6 [[0, 1], [1, 2], [3, 4]] đúng: 3 | sai: 2
4 [[0, 1]] đúng: 3 | sai: 1


**Vì sao sai:** nút không có cạnh cũng là một thành phần. Vòng ngoài phải chạy `for v in range(n)`.

### Lỗi 2: Công thức `n − len(edges)` (bỏ qua chu trình)
Xem phần test: tam giác cho `0`. **Vì sao sai:** cạnh nối hai nút đã liên thông không giảm số thành phần. **Code đúng:** chỉ trừ khi `find(a) != find(b)`.

### Lỗi 3: Giảm bộ đếm cả khi `find(a) == find(b)`
#### Code sai

In [21]:
def sai_giam_moi_canh(n, edges):
    parent = list(range(n))

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    count = n
    for a, b in edges:
        ra, rb = find(a), find(b)
        parent[ra] = rb
        count -= 1                                          # SAI: giảm cả khi ra == rb
    return count


for n, edges in [(3, [[0, 1], [1, 2], [0, 2]]), (5, [[0, 1], [1, 2], [3, 4]])]:
    print(n, edges, "đúng:", sol.countComponents(n, edges), "| sai:", sai_giam_moi_canh(n, edges))

3 [[0, 1], [1, 2], [0, 2]] đúng: 1 | sai: 0
5 [[0, 1], [1, 2], [3, 4]] đúng: 2 | sai: 2


**Vì sao sai:** chỉ gộp thành công mới giảm số thành phần; cạnh thừa không đổi gì.

### Lỗi 4: Danh sách kề một chiều
#### Code sai

In [22]:
def sai_mot_chieu(n, edges):
    adj = [[] for _ in range(n)]
    for a, b in edges:
        adj[a].append(b)                                     # SAI: thiếu adj[b].append(a)
    visited, count = [False] * n, 0
    for s in range(n):
        if visited[s]:
            continue
        count += 1
        stack = [s]
        visited[s] = True
        while stack:
            v = stack.pop()
            for u in adj[v]:
                if not visited[u]:
                    visited[u] = True
                    stack.append(u)
    return count


print("[[1,0],[1,2]] n=3 đúng:", sol.countComponents(3, [[1, 0], [1, 2]]), "| một chiều:", sai_mot_chieu(3, [[1, 0], [1, 2]]))

[[1,0],[1,2]] n=3 đúng: 1 | một chiều: 2


**Vì sao sai:** đồ thị **vô hướng**: phải thêm cả hai chiều; bản sai phụ thuộc hướng ghi cạnh.

### Lỗi 5: Không đánh dấu đã thăm (lặp vô hạn / chậm)
#### Code sai

In [23]:
def sai_khong_danh_dau(n, edges, limit=100_000):
    adj = [[] for _ in range(n)]
    for a, b in edges:
        adj[a].append(b)
        adj[b].append(a)
    stack = [0]
    steps = 0
    while stack and steps < limit:                           # limit chỉ để demo
        steps += 1
        v = stack.pop()
        for u in adj[v]:
            stack.append(u)                                  # SAI: không kiểm tra đã thăm
    return steps


print("đồ thị chỉ có 2 nút nối nhau: số bước của bản sai =", sai_khong_danh_dau(2, [[0, 1]]), "(đạt giới hạn demo, không bao giờ dừng)")

đồ thị chỉ có 2 nút nối nhau: số bước của bản sai = 100000 (đạt giới hạn demo, không bao giờ dừng)


**Vì sao sai:** hai nút nối nhau đẩy lẫn nhau mãi. Phải có `visited`.

### Lỗi 6: `find` đệ quy bị tràn khi chuỗi dài
Bản nộp "LeetCode" ở Cách 2 dùng `find` đệ quy; với `n = 2000` và cạnh sắp xếp xấu, cây DSU có thể sâu `~2000` vượt giới hạn mặc định `1000`:
#### Code sai

In [24]:
def count_dsu_recursive(n, edges):
    p = list(range(n))

    def find(x):
        if p[x] != x:
            p[x] = find(p[x])
        return p[x]

    for a, b in edges:
        pa, pb = find(a), find(b)
        if pa != pb:
            p[pa] = pb
            n -= 1
    return n


deep = [[i, i + 1] for i in range(1998)] + [[1999, 0]]
try:
    print("find đệ quy:", count_dsu_recursive(2000, deep))
except RecursionError:
    print("RecursionError (cây DSU sâu ~2000)")
print("find lặp   :", sol.countComponents(2000, deep))

RecursionError (cây DSU sâu ~2000)
find lặp   : 1


**Cách sửa:** `find` lặp (như bản nộp ở mục 9), hoặc gộp theo kích thước.

## 14. Mẹo làm nhanh & nhận dạng pattern

- Thấy "**đếm số nhóm / thành phần / đảo / mạng**" → **DSU** (đếm gộp) hoặc **DFS/BFS** (đếm số lần bắt đầu).
- **Số thành phần = `n − số lần gộp thành công`**. Cạnh thừa tạo chu trình không tính.
- Vòng ngoài luôn chạy qua **mọi nút `0..n-1`**.
- Vô hướng ⇒ **hai chiều**; có `visited`.
- `n` lớn và chuỗi dài: `find` **lặp** hoặc union by size.
- Họ hàng: **Graph Valid Tree** (261), **Number of Islands** (200), **Friend Circles / Provinces** (547), **Redundant Connection** (684), **Accounts Merge** (721).
- Thử: không cạnh, toàn nút cô lập, tam giác, đường dài, sao.

## 15. Biến thể của bài

1. **Number of Provinces** (547): đồ thị cho bằng ma trận kề.
2. **Number of Islands** (200): thành phần liên thông trên lưới.
3. **Kích thước thành phần lớn nhất** (DSU có `size`).
4. **Số thành phần sau mỗi cạnh được thêm** (online): in dãy đếm.
5. **Cạnh cần thêm để đồ thị liên thông**: `components − 1`.
#### Code demo

In [25]:
def components_after_each_edge(n, edges):
    parent = list(range(n))

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    count, history = n, []
    for a, b in edges:
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[ra] = rb
            count -= 1
        history.append(count)
    return history


def largest_component(n, edges):
    parent, size = list(range(n)), [1] * n

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    for a, b in edges:
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[ra] = rb
            size[rb] += size[ra]
    return max(size[find(v)] for v in range(n))


def edges_to_connect(n, edges):
    return sol.countComponents(n, edges) - 1


print(components_after_each_edge(5, [[0, 1], [1, 2], [3, 4], [0, 2]]), largest_component(6, [[0, 1], [1, 2], [3, 4]]), edges_to_connect(6, [[0, 1], [1, 2], [3, 4]]))
# [4, 3, 2, 2] 3 2

[4, 3, 2, 2] 3 2


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Bao đóng bắc cầu | liên thông mọi cặp | `O(V³)` | `O(V²)` | oracle |
| 2. DSU nén đường đi | gộp, đếm thành công | `O(E·α)` | `O(V)` | **bản nộp chuẩn**, cạnh đến dần |
| 3. DSU theo kích thước | nối cây nhỏ vào lớn | `O(E·α)` | `O(V)` | cần kích thước thành phần |
| 4. DFS lặp | đếm số lần bắt đầu duyệt | `O(V+E)` | `O(V+E)` | cần liệt kê nút từng thành phần |
| 5. BFS | tương tự DFS | `O(V+E)` | `O(V+E)` | cần khoảng cách trong thành phần |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: (a) `Solution().countComponents(3, [[0, 1]])`; (b) `Solution().countComponents(4, [[0, 1], [1, 2], [2, 0]])`; (c) `Solution().countComponents(1, [])`; (d) `components_after_each_edge(4, [[0, 1], [2, 3], [1, 3]])`.

**Bài 2 (tính bằng tay).** Với `n = 7`, `edges = [[0,1],[2,3],[3,4],[5,6],[4,2]]`: liệt kê các thành phần và chạy tay DSU, ghi số thành phần sau mỗi cạnh.

**Bài 3 (điền TODO).** Hoàn thiện `count_todo(n, edges)` bằng DSU.
#### Code demo

In [26]:
def count_todo(n, edges):
    parent = list(range(n))

    def find(x):
        # TODO 1: while parent[x] != x: parent[x] = parent[parent[x]]; x = parent[x]; rồi trả x
        pass

    count = n
    # TODO 2: với mỗi cạnh (a, b): ra, rb = find(a), find(b)
    # TODO 3: nếu ra != rb: parent[ra] = rb; count -= 1
    # TODO 4: trả count
    pass


# Tự làm xong rồi mới mở test:
# for n, edges, expected in examples:
#     assert count_todo(n, edges) == expected

**Bài 4 (tự viết lại).** Viết `count_components_matrix(is_connected)` (LeetCode 547) nhận **ma trận kề** `n × n` và đếm số "tỉnh" bằng DFS; đối chiếu với `Solution().countComponents` sau khi đổi ma trận thành danh sách cạnh.

**Bài 5 (biến thể / mini challenge).** Viết `num_islands(grid)` (LeetCode 200) đếm số đảo (ô `'1'` liền kề 4 hướng) bằng DSU trên chỉ số `r * n + c`, và đối chiếu với DFS.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: nút không cạnh vẫn tính; chu trình không giảm thêm. Bài 2: mỗi cạnh gộp hoặc thừa. Bài 5: mỗi ô `'1'` là một nút; nối với ô `'1'` kề.
- **Mức 2 (vừa):** Bài 1: `2`, `2`, `1`, `[3, 2, 1]`. Bài 2: `{0,1}`, `{2,3,4}`, `{5,6}` ⇒ `3` thành phần; số thành phần sau mỗi cạnh: `6, 5, 4, 3, 3`. Bài 3: khung Cách 2.
- **Mức 3 (sâu):** Bài 4: duyệt `i`, nếu chưa thăm thì `+1` và DFS theo `is_connected[i][j] == 1`. Bài 5: `index = r * n + c`; chỉ gộp với ô trên và ô trái là `'1'`; số đảo = số ô `'1'` trừ số lần gộp thành công.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [27]:
# Bài 1
assert Solution().countComponents(3, [[0, 1]]) == 2
assert Solution().countComponents(4, [[0, 1], [1, 2], [2, 0]]) == 2
assert Solution().countComponents(1, []) == 1
assert components_after_each_edge(4, [[0, 1], [2, 3], [1, 3]]) == [3, 2, 1]

# Bài 2
b2 = [[0, 1], [2, 3], [3, 4], [5, 6], [4, 2]]
assert components_after_each_edge(7, b2) == [6, 5, 4, 3, 3] and Solution().countComponents(7, b2) == 3 == oracle(7, b2)


# Bài 3
def count_todo_ans(n, edges):
    parent = list(range(n))

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    count = n
    for a, b in edges:
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[ra] = rb
            count -= 1
    return count


for n, edges, expected in examples:
    assert count_todo_ans(n, edges) == expected


# Bài 4
def count_components_matrix(is_connected):
    n = len(is_connected)
    visited = [False] * n
    count = 0
    for i in range(n):
        if visited[i]:
            continue
        count += 1
        visited[i] = True
        stack = [i]
        while stack:
            v = stack.pop()
            for j in range(n):
                if is_connected[v][j] == 1 and not visited[j]:
                    visited[j] = True
                    stack.append(j)
    return count


rng = random.Random(24)
for _ in range(100):
    n = rng.randint(1, 10)
    edges = random_graph(n, rng.randint(0, 12), rng)
    matrix = [[1 if i == j else 0 for j in range(n)] for i in range(n)]
    for a, b in edges:
        matrix[a][b] = matrix[b][a] = 1
    assert count_components_matrix(matrix) == sol.countComponents(n, edges)


# Bài 5
def num_islands(grid):
    m, n = len(grid), len(grid[0])
    parent = list(range(m * n))

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    ones, merges = 0, 0
    for r in range(m):
        for c in range(n):
            if grid[r][c] != "1":
                continue
            ones += 1
            for dr, dc in ((1, 0), (0, 1)):
                nr, nc = r + dr, c + dc
                if nr < m and nc < n and grid[nr][nc] == "1":
                    ra, rb = find(r * n + c), find(nr * n + nc)
                    if ra != rb:
                        parent[ra] = rb
                        merges += 1
    return ones - merges


def num_islands_dfs(grid):
    m, n = len(grid), len(grid[0])
    seen, count = set(), 0
    for r in range(m):
        for c in range(n):
            if grid[r][c] == "1" and (r, c) not in seen:
                count += 1
                stack = [(r, c)]
                seen.add((r, c))
                while stack:
                    x, y = stack.pop()
                    for dx, dy in ((1, 0), (-1, 0), (0, 1), (0, -1)):
                        nx, ny = x + dx, y + dy
                        if 0 <= nx < m and 0 <= ny < n and grid[nx][ny] == "1" and (nx, ny) not in seen:
                            seen.add((nx, ny))
                            stack.append((nx, ny))
    return count


for _ in range(200):
    m, n = rng.randint(1, 6), rng.randint(1, 6)
    g = [["1" if rng.random() < 0.5 else "0" for _ in range(n)] for _ in range(m)]
    assert num_islands(g) == num_islands_dfs(g)
print("PASS đáp án bài tập")

PASS đáp án bài tập


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (DSU, `find` lặp)

```python
class Solution:
    def countComponents(self, n: int, edges: List[List[int]]) -> int:
        parent = list(range(n))

        def find(x):
            while parent[x] != x:
                parent[x] = parent[parent[x]]
                x = parent[x]
            return x

        components = n
        for a, b in edges:
            ra, rb = find(a), find(b)
            if ra != rb:
                parent[ra] = rb
                components -= 1
        return components
```

**Checklist**

- [ ] Bắt đầu `components = n` (mỗi nút một thành phần)?
- [ ] Chỉ **giảm** khi `find(a) != find(b)` (gộp thành công)?
- [ ] Nếu dùng DFS/BFS: vòng ngoài chạy **mọi nút** `0..n-1`, có `visited`, cạnh **hai chiều**?
- [ ] Không dùng công thức `n − len(edges)` (sai khi có chu trình)?
- [ ] `find` lặp / union by size để tránh tràn đệ quy?
- [ ] Đã thử không cạnh, nút cô lập, tam giác, đường dài, `n = 2000`?
- [ ] Đã đối chiếu với brute force bao đóng bắc cầu trên đồ thị ngẫu nhiên nhỏ?
- [ ] Skeleton đúng `class Solution` / `def countComponents(self, n: int, edges: List[List[int]]) -> int`?